# Machine Learning Modeling Pipeline

In this phase, we will build a robust, end-to-end Machine Learning pipeline. Our workflow includes:
1. **Data Preprocessing:** Handling missing values, scaling numerical features, and encoding categorical features using `ColumnTransformer`.
2. **Dimensionality Reduction:** Applying PCA (Principal Component Analysis) to handle the high number of features.
3. **Model Training & Tuning:** Utilizing tree-based models (Random Forest, XGBoost) and optimizing them via `GridSearchCV` with K-Fold Cross Validation.

In [ ]:
import pandas as pd
import numpy as np

# Pipeline and Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.decomposition import PCA

# Model Selection and Metrics
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Models 
from sklearn.ensemble import RandomForestRegressor

## 1. Data Preparation and Train-Test Split

Before touching the data, we must split it into training and testing sets. This prevents data leakage, ensuring our imputers and scalers only learn from the training data. We will also drop the `Id` column as it holds no predictive value.

In [ ]:
# Load the data
df = pd.read_csv("../data/raw/train.csv")

# Drop the 'Id' column
df.drop("Id", axis=1 , inplace=True)

# Seperate features (X) and target (y)
X = df.drop('SalePrice', axis=1)
y = df.SalePrice

# Split the data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (1168, 79)
Testing features shape: (292, 79)


## 2. Building the Preprocessing Pipeline

We will create two separate pipelines:
*   **Numerical Pipeline:** Uses `KNNImputer` to fill missing values based on similar rows, followed by `RobustScaler` to scale data without being heavily influenced by extreme outliers.
*   **Categorical Pipeline:** Uses `SimpleImputer` to fill missing values with a constant ('Missing'), followed by `OneHotEncoder` (ignoring unknown categories found in the test set).

These are then combined using `ColumnTransformer`.

In [11]:
# Identify numerical and categorical columns
num_cols = X_train.select_dtypes(include=["float64","int64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=object).columns.tolist()

# 1. Numerical Pipeline
num_pipeline = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", RobustScaler())
])

# 2. Categorical Pipeline
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant",fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

# 3. Combine into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
])

C:\Users\Tolga\AppData\Local\Temp\ipykernel_26536\878346013.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=object).columns.tolist()


## 3. The Final Pipeline

Now we combine our data preprocessing steps with Dimensionality Reduction (PCA) and our predictive model. 

*   **PCA (Principal Component Analysis):** Since `OneHotEncoder` will create many new columns (high dimensionality), PCA will reduce the number of features while retaining most of the variance. 
*   **Model:** We will start with a `RandomForestRegressor`.

*Tree-based models often perform well without PCA, but including it demonstrates a complete, industry-standard machine learning pipeline.*

## 4. Hyperparameter Tuning with Grid Search & K-Fold Cross Validation

Instead of guessing the best parameters for our model and PCA, we will use `GridSearchCV`. It will systematically test different combinations of parameters.

To ensure our model evaluates these combinations robustly and doesn't overfit, we will use **K-Fold Cross Validation** (`KFold`). It splits the training data into 'K' different subsets, training on K-1 folds and validating on the remaining fold.

## 5. Training and Evaluation

Now we execute the Grid Search. This step will apply the entire pipeline (Imputing -> Scaling/Encoding -> PCA -> Modeling) to every fold in our cross-validation setup for every parameter combination.